In [27]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END,MessagesState
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model =  ChatGroq(
        model = "llama-3.1-8b-instant"
    )

In [33]:
def callModel(state: MessagesState) -> str:
    response = model.invoke(state["messages"])
    return {"messages": [response]}

In [34]:
builder = StateGraph(MessagesState)
builder.add_node("call_LLM",callModel)
builder.add_edge(START, "call_LLM")
builder.add_edge("call_LLM", END)


In [35]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [36]:
config = {"configurable": {"thread_id": "thread-1" }}
graph.invoke({"messages": [{"role": "user", "content": "My name is Amna Ali"}]}, config=config)  

{'messages': [HumanMessage(content='My name is Amna Ali', additional_kwargs={}, response_metadata={}, id='71e10506-9548-44eb-af16-a035cc5e4f8f'),
  AIMessage(content="Hello Amna Ali, it's nice to meet you. Is there something I can help you with today, or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 41, 'total_tokens': 71, 'completion_time': 0.032056354, 'completion_tokens_details': None, 'prompt_time': 0.001964872, 'prompt_tokens_details': None, 'queue_time': 0.00524703, 'total_time': 0.034021226}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d8647-aa37-7b41-bbf8-4ecde0612040-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 30, 'total_tokens': 71})]}

In [37]:

graph.invoke({"messages": [{"role": "user", "content": "What is my name"}]}, config=config)  

{'messages': [HumanMessage(content='My name is Amna Ali', additional_kwargs={}, response_metadata={}, id='71e10506-9548-44eb-af16-a035cc5e4f8f'),
  AIMessage(content="Hello Amna Ali, it's nice to meet you. Is there something I can help you with today, or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 41, 'total_tokens': 71, 'completion_time': 0.032056354, 'completion_tokens_details': None, 'prompt_time': 0.001964872, 'prompt_tokens_details': None, 'queue_time': 0.00524703, 'total_time': 0.034021226}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d8647-aa37-7b41-bbf8-4ecde0612040-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 30, 'total_tokens': 71}),
  HumanMessage(content='What is my name', additional_kwargs={}, respon

In [38]:
snap = graph.get_state(config)
vals = snap.values
for m in vals.get("messages",[]):
    print(m.content)

My name is Amna Ali
Hello Amna Ali, it's nice to meet you. Is there something I can help you with today, or would you like to chat?
What is my name
Your name is Amna Ali.
